In [4]:
%reload_ext autoreload
%autoreload 2

# M5 MLForecast

## Imports

In [5]:
import time
import os
import numpy as np
import warnings
import multiprocessing
import pandas as pd
from pathlib import Path

id_col = "id"
time_col = "date"
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

PATH_INPUT = Path("data/m5/datasets")
N_JOBS = multiprocessing.cpu_count()

HORIZON = 28  # How many days into the future to predict
VAL_DAYS = 0  # How many days of data to hold-out for validatioin (set to 0 for live)
LAST_N_DAYS = 400  # How many days to keep for training

if not PATH_INPUT.exists():
    raise FileNotFoundError("Please download the M5 dataset first by executing `src/generate_data.py`")
    
os.environ["NIXTLA_ID_AS_COL"] = '1'
start_time = time.time()
warnings.simplefilter("ignore")

# Helper Functions

In [6]:
start_time = time.time()

from typing import List
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import ExpandingMean, RollingMean, SeasonalRollingMean
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mse, mase, mape, mae, smape
from utilsforecast.plotting import plot_series
from utilsforecast.preprocessing import fill_gaps
from src.process import *

def is_first_or_fifteenth(dates):
    """Date is the 1st or 15th of the month"""
    return dates.day.isin([1, 15])

def even_day(dates):
    """Day of month is even"""
    return dates.day % 2 == 0

def month_start_or_end(dates):
    """Date is month start or month end"""
    return dates.is_month_start | dates.is_month_end

def is_monday(dates):
    """Date is monday"""
    return dates.dayofweek == 0

def downcast_id_col(df: pd.DataFrame, 
                    df_ids: pd.DataFrame, 
                    id_col: str = id_col):
    """Downcast df[id_col] to integer.
    """
    df[id_col] = df[id_col].map(df_ids.set_index("id")["idx"].to_dict())

def is_business_day(dates):
    """Date is a business day (Monday to Friday)"""
    return dates.dt.dayofweek < 5  # Monday=0, Sunday=6

def is_quarter_start(dates):
    """Date is quarter start"""
    return dates.dt.is_quarter_start

def even_day(dates):
    """Day of month is even"""
    return dates.dt.day % 2 == 0

def month_start_or_end(dates):
    """Date is month start or month end"""
    return dates.dt.is_month_start | dates.dt.is_month_end

def is_monday(dates):
    """Date is Monday"""
    return dates.dt.dayofweek == 0

def is_weekend(dates):
    """Date is on a weekend (Saturday or Sunday)"""
    return dates.dt.dayofweek >= 5

def is_week_start(dates):
    """Date is the start of the week (Monday)"""
    return dates.dt.dayofweek == 0

def is_week_end(dates):
    """Date is the end of the week (Sunday)"""
    return dates.dt.dayofweek == 6

def is_year_start_or_end(dates):
    """Date is year start or year end"""
    return dates.dt.is_year_start | dates.dt.is_year_end

def is_leap_year(dates):
    """Date is in a leap year"""
    return dates.dt.is_leap_year

def days_until_month_end(dates):
    """Number of days until the end of the month"""
    month_end = dates + pd.offsets.MonthEnd(0)
    return (month_end - dates).dt.days

def is_fiscal_year_end(dates, month=12):
    """Date is the end of the fiscal year
    Args:
        dates (pd.Series): Series of datetime objects.
        month (int): Month considered as the fiscal year-end.
    """
    return (dates.dt.month == month) & dates.dt.is_month_end

def day_of_quarter(dates):
    """Day number within the quarter"""
    quarter_start = dates.dt.to_period('Q').start_time
    return (dates - quarter_start).dt.days + 1

def week_of_month(dates):
    """Week number within the month"""
    return ((dates.dt.day - 1) // 7) + 1

def is_special_date(dates, special_dates):
    """Date matches any in a list of special dates
    Args:
        dates (pd.Series): Series of datetime objects.
        special_dates (list or set): Special dates to check against.
    """
    special_dates = pd.to_datetime(special_dates)
    return dates.isin(special_dates)


def evaluate_cross_validation(df: pd.DataFrame,
                              metric: callable, 
                              id_col: str = 'unique_id',
                              time_col: str = "ds",
                             ) -> pd.DataFrame:
    """Evaluate cross-validation results.
    """
    models = df.drop(columns=[id_col, 'ds', 'cutoff', 'y']).columns.tolist()
    evals = []
    # Calculate loss for every unique_id and cutoff.    
    for cutoff in df['cutoff'].unique():
        eval_ = evaluate(df[df['cutoff'] == cutoff],
                         metrics=[metric],
                         models=models,
                         id_col=id_col,
                         time_col=time_col)
        evals.append(eval_)
    evals = pd.concat(evals)
    evals = evals.groupby(id_col).mean(numeric_only=True) # Averages the error metrics for all cutoffs for every combination of model and unique_id
    evals['best_model'] = evals.idxmin(axis=1)
    return evals


def get_best_model_forecast(forecasts_df: pd.DataFrame,
                            evaluation_df: pd.DataFrame, 
                            id_col: str = "unique_id") -> pd.DataFrame:
    """Create Production Forecast DataFrame.

    returns: df Dataframe with columns:
        id_col, ds, best_model, forecast.
    """
    df = forecasts_df.set_index([id_col, "ds"]).stack().to_frame().reset_index(level=2)
    df.columns = ['model', 'best_model_forecast']
    df = df.join(evaluation_df[['best_model']])

    df = df[df["best_model"] == df["model"]].reset_index().drop("model", axis=1)
    return df

# Load Inputs

In [7]:
dfids = get_dfids()
df = pd.read_parquet("data/train.snap.parquet")
df = filter_data(df, LAST_N_DAYS)

2025-09-01 19:14:45.549 | INFO     | src.process:filter_data:77 - Rows of input data: 46,796,220 for 1,941 days of Sales
2025-09-01 19:14:45.550 | INFO     | src.process:filter_data:80 - Drop training data older than 400 days old
2025-09-01 19:14:47.663 | INFO     | src.process:filter_data:90 - Rows of processed input data: 12,159,132


# LightGbm CV

Fit the above MLForecast with lgb.LGBMRegressor parameters `lgb_params` specified.

- FREQ: `D` daily
- LAG_FEATURES: Calculates a list of `n` period with (freq) lag (i.e. 1, 7 would be y_1 and y_7 equal to yesterday and today. 
- LAG_TRANSFORM_DICT: Specify dictionary of tranformations to apply against `LAG_FEATURES`

- `lgb_params`: parameter dictionary for lightgbm
    - `num_leaves` and `n_estimators` are the key ones handling regularization and complexity

In [8]:
%%time


from mlforecast.lag_transforms import ExpandingMean, RollingMean, SeasonalRollingMean, ExpandingStd

lgb_params = {
    'verbose': 1,
    'num_threads': 4,
    'force_col_wise': True,
    'num_leaves': 256,
    'n_estimators': 1000,  # 500
}

# LAG_FEATURES = [1, 2, 4, 7, 14, 21, 28, 35, 42, 56, 84, 168]

# LAG_TRANSFORM_DICT = {
#     1:  [ExpandingMean(), ExpandingStd()],
#     7:  [RollingMean(7), SeasonalRollingMean(7, 1), ExpandingStd()],
#     14: [RollingMean(14), SeasonalRollingMean(14, 1)],
#     28: [RollingMean(28), SeasonalRollingMean(28, 1)],
#     84: [RollingMean(84), SeasonalRollingMean(84, 1)],  # Quarterly trend
# }

LAG_FEATURES = [1, 2, 3, 7, 14, 28, ]

LAG_TRANSFORM_DICT = {
    1:  [ExpandingMean()],
    7:  [RollingMean(7), SeasonalRollingMean(7, 1)],    # Capture weekly trend ajusting for seasonality (day of week effects)
    14: [RollingMean(14), SeasonalRollingMean(14, 1)],  # Capture bi-weekly trend adjusting for seasonality (i.e. payday)
    28: [RollingMean(28), SeasonalRollingMean(28, 1)],  # Capture Monthly trend adjusting for monthly seasonality
}

# List of built-in date features to compute with MLForecast.date_features
DATE_FEATURE_LIST = [
        'year', 'month', 'day', 'day_of_week', 'quarter', 'week',
        'is_month_start', 'is_month_end', 'is_quarter_start', 'is_quarter_end',
        is_first_or_fifteenth,
    ]

fcst = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq='D',
    lags=LAG_FEATURES,
    lag_transforms = LAG_TRANSFORM_DICT,
    date_features = DATE_FEATURE_LIST,
    num_threads=4,
)



PARAMS = fcst.models["LGBMRegressor"].get_params()

# lgb_cv = fcst.cross_validation(h=HORIZON,
#                                n_windows=1,
#                                df=df,
#                                static_features=id_cols,
#                                id_col=id_col, time_col=time_col
#                               )

CPU times: total: 0 ns
Wall time: 997 μs


In [9]:
PARAMS

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.1,
 'max_depth': -1,
 'min_child_samples': 20,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 1000,
 'n_jobs': None,
 'num_leaves': 256,
 'objective': None,
 'random_state': None,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0,
 'verbose': 1,
 'num_threads': 4,
 'force_col_wise': True}

# Fit LightGbm on Full Data

In [10]:
%%time

lgb_start_time = time.time()

last_date_train = df['date'].max()
last_wmyrwk = df[df['date'] < last_date_train]['wm_yr_wk'].max()
logger.info(f"""Begin fitting LightGbm with {df[id_col].nunique():,d} ids data before: {last_date_train.date()}""")

logger.info(f"""LGB Fit Parameters:\n{PARAMS}""")
fcst.fit(
    df,
    id_col=id_col,
    time_col=time_col,
    target_col='y',
    static_features=id_cols,
)

logger.info(f"""Finished fitting LightGbm on {df["id"].nunique():,d} time-series in {time.time()- lgb_start_time:,.2f} seconds.""")

2025-09-01 19:14:59.689 | INFO     | __main__:<module>:5 - Begin fitting LightGbm with 30,490 ids data before: 2016-05-22
2025-09-01 19:14:59.690 | INFO     | __main__:<module>:7 - LGB Fit Parameters:
{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.1, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 1000, 'n_jobs': None, 'num_leaves': 256, 'objective': None, 'random_state': None, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbose': 1, 'num_threads': 4, 'force_col_wise': True}


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Total Bins 34876
[LightGBM] [Info] Number of data points in the train set: 10482182, number of used features: 40
[LightGBM] [Info] Start training from score 1.317613


KeyboardInterrupt: 

In [ ]:
fcst.models_

In [ ]:
## Create Future Regressors

cal = load_calendar(PATH_INPUT)
prices = load_prices(PATH_INPUT)

X_df = create_future_features(last_date_train, cal, prices, HORIZON)
X_df = X_df.merge(cal[["date", "d", "wm_yr_wk"]], on='date', how='left')

In [ ]:
%%time
preds = fcst.predict(HORIZON, X_df=X_df)
logger.info(f"""Finished generating {HORIZON:,d} days of predictions for {preds["id"].nunique():,d} time-series in {time.time()- lgb_start_time:,.2f} seconds.""")

In [ ]:
preds.tail()


# Create and Score Submission

In [33]:
def make_submission(preds: pd.DataFrame,
                    h: int) -> pd.DataFrame:
    wide = preds.pivot_table(index='id',
                             columns='date',
                             observed=True)
    wide.columns = [f'F{i+1}' for i in range(h)]
    # wide.columns = [f'd_{TEST_START + i}' for i in range(h)]
    wide.columns.name = None
    wide.index.name = 'id'
    return wide

In [34]:
from datasetsforecast.m5 import M5Evaluation

dfids = get_dfids()
submission = make_submission(preds[["id", time_col, "LGBMRegressor"]], HORIZON)

df_score = M5Evaluation.evaluate("data", submission.join(dfids.set_index("id")))
model_score = df_score["wrmsse"].mean()

model_avg = df_score[~df_score.index.isin(["Total"])]["wrmsse"].mean()
model_score = df_score.loc["Total"].iloc[0]

In [35]:
from datasetsforecast.m5 import M5Evaluation

dfids = get_dfids()
submission = make_submission(preds[["id", time_col, "LGBMRegressor"]], HORIZON)

df_score = M5Evaluation.evaluate("data", submission.join(dfids.set_index("id")))
model_score = df_score["wrmsse"].mean()

model_avg = df_score[~df_score.index.isin(["Total"])]["wrmsse"].mean()
model_score = df_score.loc["Total"].iloc[0]

df_score

,wrmsse
Total,0.925705
Level1,0.878453
Level2,0.870340
Level3,0.878458
Level4,0.894320
Level5,0.944474
Level6,0.904169
Level7,0.949695
Level8,0.888007
Level9,0.928945


In [36]:
# Save Submission
submission.to_parquet("data/submission_mlforecast.snap.parquet")

In [20]:
pd.read_parquet("data/submission_mlforecast.snap.parquet")

,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
id,,,,,,,,,,,,,,,,,,,,,
FOODS_1_001_CA_1_evaluation,0.688216,0.736647,0.755982,0.704002,0.894199,1.099872,1.049504,0.766041,0.675283,0.732291,...,0.833224,0.993821,0.904707,0.692947,0.692947,0.692947,0.698467,0.786863,0.968618,0.973294
FOODS_1_001_CA_2_evaluation,0.875772,0.986453,0.982452,1.000675,1.084578,1.336825,1.283334,1.012339,1.018205,1.023773,...,1.130471,1.326814,1.266588,0.946207,0.946207,0.936577,0.930872,1.074646,1.354231,1.285509
FOODS_1_001_CA_3_evaluation,0.799446,0.818329,0.938102,0.897997,1.474033,1.856071,1.842591,0.960248,0.870696,0.924341,...,1.203108,1.926519,1.835969,0.911027,0.906664,1.018074,0.979063,1.523770,1.931197,1.909920
FOODS_1_001_CA_4_evaluation,0.224043,0.280978,0.354310,0.382192,0.380039,0.391107,0.401954,0.375859,0.382192,0.381147,...,0.416076,0.424803,0.424803,0.387890,0.387890,0.400944,0.396887,0.406239,0.422335,0.422335
FOODS_1_001_TX_1_evaluation,0.607613,0.569456,0.822326,0.770904,0.848629,0.921484,0.994264,0.687975,0.692647,0.677248,...,0.904574,1.065380,1.065380,0.815417,0.785603,0.770564,0.770611,0.801628,0.801171,0.829056
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HOUSEHOLD_2_516_TX_2_evaluation,0.260402,0.269805,0.276802,0.267303,0.286845,0.308733,0.342579,0.310862,0.297669,0.297669,...,0.329196,0.352764,0.345202,0.317021,0.307855,0.300292,0.300292,0.316190,0.338079,0.363611
HOUSEHOLD_2_516_TX_3_evaluation,0.209528,0.255533,0.245729,0.241227,0.276032,0.296988,0.340711,0.280341,0.281222,0.325344,...,0.295474,0.338734,0.327888,0.281612,0.281612,0.287478,0.287478,0.306320,0.350488,0.377180
HOUSEHOLD_2_516_WI_1_evaluation,0.144665,0.139257,0.174288,0.193937,0.269825,0.266891,0.266891,0.199664,0.199664,0.199664,...,0.303521,0.329375,0.329375,0.233360,0.233360,0.233360,0.233360,0.314368,0.329375,0.329375


In [38]:
print(f"""MLForecast wrmsse: {model_score:.4f}""")
print(f"""Notebook finished in {(time.time() - start_time) / 60:.2f}m""")

MLForecast wrmsse: 0.9257
Notebook finished in 15.40m


In [37]:
# ### Parameter Optimization

# import optuna
# import lightgbm as lgb

# def objective(trial):
#     params = {
#         'num_leaves': trial.suggest_int('num_leaves', 31, 512),
#         'learning_rate': trial.suggest_loguniform('learning_rate', 0.005, 0.1),
#         'max_depth': trial.suggest_int('max_depth', 3, 16),
#         'n_estimators': trial.suggest_int('n_estimators', 100, 500),
#         'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.6, 1.0),
#         'subsample': trial.suggest_uniform('subsample', 0.6, 1.0),
#         'min_child_weight': trial.suggest_loguniform('min_child_weight', 0.1, 10),
#     }
#     model = lgb.LGBMRegressor(**params)
#     fcst = MLForecast(models=[model], freq='D', lags=NEW_LAG_FEATURES, lag_transforms=NEW_LAG_TRANSFORM_DICT)
#     fcst.fit(df, id_col=id_col, time_col=time_col, target_col='y', static_features=id_cols)
#     preds = fcst.predict(HORIZON)
#     wrmse_score = evaluate_cross_validation(preds, metric=wrmse)  # Custom WRMSSE function
#     return wrmse_score

# # study = optuna.create_study(direction='minimize')
# # study.optimize(objective, n_trials=20)
# # best_params = study.best_params


In [67]:
# ### Hierarchical Reconciliation

# from hierarchicalforecast.core import HierarchicalReconciliation
# from hierarchicalforecast.methods import BottomUp, TopDown, MiddleOut, MinTrace, OptimalCombination, ERM
# from hierarchicalforecast.evaluation import HierarchicalEvaluation